# Lab 3

In [5]:
pip install nltk

  Using cached nltk-3.10.3-py3-none-any.whl.metadata (3.2 kB)
  Using cached defusedxml-0.7.1-py2.py3-none-any.whl.metadata (32 kB)
Using cached nltk-3.10.3-py3-none-any.whl (1.8 MB)
Using cached defusedxml-0.7.1-py2.py3-none-any.whl (25 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [nltk]━━━━━━ 1/2 [nltk]
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import tarfile

paths = [Path('../sts/resources/test-gold.tgz'), Path('sts/resources/test-gold.tgz'), Path('test-gold.tgz')]
path = next((p for p in paths if p.is_file()), None)
if path is None:
    raise FileNotFoundError()

with tarfile.open(path) as tar:
    pairs = [line.split('\t') for line in tar.extractfile('test-gold/STS.input.SMTeuroparl.txt').read().decode('utf-8').splitlines()]
    gs = [float(line) for line in tar.extractfile('test-gold/STS.gs.SMTeuroparl.txt').read().decode('utf-8').splitlines()]

assert len(pairs) == len(gs)
assert all(len(pair) == 2 for pair in pairs)
len(pairs), len(gs)

(459, 459)

In [6]:
print(f"{'#':<3} {'sentence 1':<65} sentence 2")
for i, (a, b) in enumerate(pairs[:5], 1):
    print(f"{i:<3} {a[:62]:<65} {b[:62]}")

#   sentence 1                                                        sentence 2
1   The leaders have now been given a new chance and let us hope t    The leaders benefit aujourd' hui of a new luck and let's let t
2   Amendment No 7 proposes certain changes in the references to p    Amendment No 7 is proposing certain changes in the references 
3   Let me remind you that our allies include fervent supporters o    I would like to remind you that among our allies, there are st
4   The vote will take place today at 5.30 p.m.                       The vote will take place at 5.30pm
5   The fishermen are inactive, tired and disappointed.               The fishermen are inactive, tired and disappointed.


## Words

In [7]:
import re
from nltk.metrics import jaccard_distance

unique_sentences = list(dict.fromkeys(sentence for pair in pairs for sentence in pair))
word_pattern = re.compile(r'\b\w+\b')
tokens = {sentence: word_pattern.findall(sentence) for sentence in unique_sentences}
word_sets = {sentence: {word.casefold() for word in words} for sentence, words in tokens.items()}
word_distances = [jaccard_distance(word_sets[a], word_sets[b]) for a, b in pairs]

print(f'{len(pairs)} pairs; {2 * len(pairs)} sentence occurrences; {len(unique_sentences)} unique sentences')

459 pairs; 918 sentence occurrences; 338 unique sentences


## Lemmas

In [8]:
import nltk
from functools import cache
from nltk.stem import WordNetLemmatizer

resources = {
    'averaged_perceptron_tagger_eng': 'taggers/averaged_perceptron_tagger_eng/',
    'wordnet': 'corpora/wordnet/',
}
for package, resource in resources.items():
    try:
        nltk.data.find(resource)
    except LookupError:
        nltk.download(package, quiet=True, raise_on_error=True)

lemmatizer = WordNetLemmatizer()
wordnet_pos = {'N': 'n', 'V': 'v', 'J': 'a', 'R': 'r'}

@cache
def lemma(word, pos):
    return lemmatizer.lemmatize(word, pos) if pos else word

/home/biel/miniconda3/envs/ahlt/lib/python3.11/site-packages/nltk/downloader.py:1076: UserWarning: NLTK will not authorize the non-private download directory '/home/biel/nltk_data': it (or an ancestor) is world- or group-writable, so another local user could plant files there. Choose a private location such as ~/nltk_data.
  for msg in self.incr_download(info_or_id, download_dir, force):


In [9]:
tagged_sentences = nltk.pos_tag_sents(list(tokens.values()))
lemma_sets = {
    sentence: {lemma(word.casefold(), wordnet_pos.get(tag[0])) for word, tag in tagged}
    for sentence, tagged in zip(unique_sentences, tagged_sentences)
}
lemma_distances = [jaccard_distance(lemma_sets[a], lemma_sets[b]) for a, b in pairs]

len(word_distances), len(lemma_distances)

(459, 459)

## Comparison

In [10]:
print(f"{'pair':<6} {'gold':<8} {'word distance':<16} Lemma distance")
for i in range(10):
    print(f"{i + 1:<6} {gs[i]:<8.3f} {word_distances[i]:<16.3f} {lemma_distances[i]:.3f}")

pair   gold     word distance    Lemma distance
1      4.500    0.667            0.667
2      5.000    0.231            0.083
3      4.250    0.619            0.619
4      4.500    0.417            0.417
5      5.000    0.000            0.000
6      5.000    0.632            0.632
7      4.667    0.706            0.706
8      5.000    0.250            0.250
9      5.000    0.714            0.714
10     5.000    0.500            0.500


In [11]:
from statistics import correlation

r_words = correlation(gs, word_distances)
r_lemmas = correlation(gs, lemma_distances)

print(f"{'Representation':<16} Pearson (distance)")
print(f"{'Words':<16} {r_words:.6f}")
print(f"{'Lemmas':<16} {r_lemmas:.6f}")

Representation   Pearson (distance)
Words            -0.485346
Lemmas           -0.482548


The word correlation differs from session 2 (−0.450) because there we used `nltk.word_tokenize`, case-sensitive and keeping punctuation tokens. Here words are lowercased and only alphanumeric tokens are kept (regex), which gives −0.485. Both words and lemmas use this same tokenization, so the comparison between them is fair.

## Justification

Words perform slightly better on this dataset (but really similar) pearson is −0.485346 for word distances and −0.482548 for lemma distances. Both correlations are negative because gold scores measure similarity and Jaccard measures distance, using 1 − distance gives +0.485346 and +0.482548.

Lemmas can help when different word forms express the same idea. For example in pair 2, “proposes” and “proposing” both become “propose”, reducing the distance from 0.231 to 0.083. They do not always help, lemmatization can remove useful grammatical distinctions, POS tagging can make mistakes, and Jaccard still misses synonyms and word order, basically it does not guarantee improvement for every pair. For example in pair 198 both sentences have the same words in a different order, so the word distance is 0.000, but the tagger labels “tired” and “disappointed” as adjectives in one sentence and as verbs in the other, so only one side becomes “tire” and “disappoint” and the lemma distance goes up to 0.444.